In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer, CrossEncoder
from qdrant_client import QdrantClient, models
from pathlib import Path
import uuid
import gc
from docling.document_converter import DocumentConverter
from docling.chunking import HybridChunker

In [2]:
encoder = SentenceTransformer(
    "Qwen/Qwen3-Embedding-0.6B",
    device="cpu"
)

client = QdrantClient(path="../saves/VScatalogo") # Carrega vectorstore em disco

In [3]:
file_path = "/home/marcos_manhaes/Documentos/edital_petrobras.pdf"
file_path = "/home/migueldcarvalho/Downloads/edital_petrobras.pdf"
#file_path = "/home/migueldcarvalho/Downloads/UERJ_2026.pdf"
collection_name = Path(file_path).name

In [4]:
def embed_texts(texts, encoder, batch_size=5):
    vectors = []
    print(f"Tamanho do texto: {len(texts)}")
    for i in range(0, len(texts), batch_size):
        print(i)
        batch = texts[i:i + batch_size]
        emb = encoder.encode(batch, convert_to_numpy=True)
        vectors.extend(emb)
        del batch, emb
        gc.collect()
    return vectors

def external_doc(state):
    client = state["client"]
    encoder = state["encoder"]

    print(f"Convertendo documento: {file_path}")
    converter = DocumentConverter()
    result = converter.convert(file_path)
    document = result.document
    print("Extraindo conteúdo...")
    
    chunker = HybridChunker(
        max_tokens=500,
        merge_peers=True
    )
    
    docling_chunks = list(chunker.chunk(document))
    
    texts = []
    chunks_metadata = []
    
    for i, chunk in enumerate(docling_chunks):
        text = chunk.text
        metadata = {
            "source": file_path,
            "chunk_index": i,
            "page_range": chunk.meta.page_range if hasattr(chunk.meta, 'page_range') else None,
            "headings": chunk.meta.headings if hasattr(chunk.meta, 'headings') else []
        }
        
        texts.append(text)
        chunks_metadata.append(metadata)
    
    print(f"Total de chunks criados: {len(texts)}")

    vectors = embed_texts(texts, encoder, batch_size=5)

    if not client.collection_exists(collection_name=collection_name):
        client.create_collection(
            collection_name=collection_name,
            vectors_config=models.VectorParams(
                size=encoder.get_sentence_embedding_dimension(),
                distance=models.Distance.COSINE,
            ),
        )
        
    points = []
    for i, text in enumerate(texts):
        
        if chunks_metadata:
            payload = {
                "page_content": text,
                "metadata": chunks_metadata[i],
                "chunk_index": i
            }
        else:
            payload = {
                "page_content": text,
                "metadata": {"source": file_path},
                "chunk_index": i
            }
        
        points.append(
            models.PointStruct(
                id=str(uuid.uuid4()),
                vector=vectors[i].tolist(),
                payload=payload
            )
        )

    batch_size = 100
    for i in range(0, len(points), batch_size):
        batch = points[i:i + batch_size]
        client.upsert(
            collection_name=collection_name,
            points=batch
        )
        print(f"Inseridos {i + len(batch)} de {len(points)} pontos")
        
        del batch
        gc.collect()

In [ ]:
state = {
    "client": client,
    "encoder": encoder
}

if not client.collection_exists(collection_name=collection_name):
    external_doc(state)
else:
    print(f"Coleção '{collection_name}' já existe. Pulando etapa de indexação.")


Coleção 'edital_petrobras.pdf' já existe. Pulando etapa de indexação.


In [12]:
query = "Quais são os requisitos para tomar posse do cargo?"
#query = "I would like information regarding the requirements for occupational nursing."
#query = "Quais são os requisitos de formação para a ênfase em Manutenção – Caldeiraria no concurso da Petrobras?"
query = "isenção da taxa de inscrição"
query = "enfase em enfermagem do trabalho"
query = "Tenho filhos, posso deixa-los em algum local durante a prova"
query = "Qual é a banca responsavel pelo concurso?"

print(f"\nConsultando: {query}")

hits = client.search(
    collection_name=collection_name,
    query_vector=encoder.encode(query).tolist(),
    limit=20  # Aumentado para ter mais candidatos ao re-ranking
)

print(f"Recuperados {len(hits)} resultados iniciais")


scored_hits = [(hit.score, hit) for hit in hits]
scored_hits.sort(reverse=True)

print("\n--- TOP 5 RESULTADOS APÓS RE-RANKING ---\n")
for i, (score, hit) in enumerate(scored_hits[:5]):
    print(f"Resultado #{i+1}")
    print(f"Score (cross-encoder): {score:.4f}")
    print(f"Score original: {hit.score:.4f}")
    
    # Mostrar metadados enriquecidos se disponíveis
    if 'headings' in hit.payload.get('metadata', {}):
        headings = hit.payload['metadata']['headings']
        if headings:
            print(f"Seção: {' > '.join(headings)}")
    
    if 'page_range' in hit.payload.get('metadata', {}):
        print(f"Páginas: {hit.payload['metadata']['page_range']}")
    
    print("\nConteúdo:")
    print(hit.payload['page_content'][:1000] + "..." if len(hit.payload['page_content']) > 1000 else hit.payload['page_content'])
    print("-" * 80)
    print()


Consultando: Qual é a banca responsavel pelo concurso?
Recuperados 20 resultados iniciais

--- TOP 5 RESULTADOS APÓS RE-RANKING ---

Resultado #1
Score (cross-encoder): 0.4085
Score original: 0.4085

Conteúdo:
homologação divulgada no DOU.  
13.10 Todas as despesas decorrentes da participação em qualquer fase deste processo seletivo 
público serão de inteira responsabilidade da pessoa candidata.  
13.11 A pessoa candidata deverá manter atualizados seus dados pessoais e seu endereço perante 
o Cebraspe enquanto estiver participando do processo seletivo público, por meio de requerimento 
a ser enviado à Central de Atendimento ao Candidato do Cebraspe, na forma dos subitens 13.7 ou 
13.8 deste edital, conforme o caso, e perante a Petrobras, após a homologação do resultado final, 
desde que aprovad a, conforme procedimento a seguir. São de exclusiva responsabilidade d a
--------------------------------------------------------------------------------

Resultado #2
Score (cross-encoder): 0.

/tmp/ipykernel_190730/2171543805.py:11: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  hits = client.search(
